# Bunch Queue Theory — V11 Comprehensive Aggression Sweep

**Author:** Jeffrey Curry — Retired Intel Engineer | Former CMG Reviewer  
**Date:** July 2026  
**Version:** V11 — Three-Run Validated

---

## Overview

V11 is the primary validation notebook for Bunch Queue Theory. It conducts a comprehensive sweep across:

- **10 aggression levels:** 5%, 15%, 30%, 60%, 65%, 75%, 80%, 85%, 90%, 95%
- **5 scales:** 100, 500, 1000, 2500, 5000 agents
- **60 runs per scenario** — three independent trials confirm reproducibility

### New Metrics Introduced in V11

- **CPU idle rate** — fraction of scheduler cycles where resources go unclaimed
- **Standard deviation** — variability signal for thrashing onset detection
- **Per-run tracking** — enables statistical confidence intervals

### Key Findings

| Finding | Result |
|---|---|
| Optimal aggression zone | 85-90% across all scales |
| Passive latency improvement | 15-26% over orderly baseline |
| CPU idle reduction | 11-26 percentage points |
| Thrashing signal | Std dev triples above 75% aggression |
| Thrashing boundary | Between 90% and 95% aggression |
| Results confirmed | Three independent 60-run trials |

### Simulation Environment

- Python 3.12, Mesa 3.5.1
- Ubuntu 24.04 LTS VM
- Results auto-saved to JSON after each scale


In [ ]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
from enum import Enum
import json
import time
import sys
import warnings
warnings.filterwarnings('ignore')

# ── Agent Types ───────────────────────────────
class AgentType(Enum):
    PASSIVE    = 1
    NORMAL     = 2
    AGGRESSIVE = 3

TYPE_COEFFICIENTS = {
    AgentType.PASSIVE:    0.3,
    AgentType.NORMAL:     0.7,
    AgentType.AGGRESSIVE: 1.0,
}

# ── Publication Plot Style ────────────────────
plt.rcParams.update({
    'font.size':         10,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'legend.fontsize':    9,
    'font.family':       'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'lines.linewidth':   2,
    'lines.markersize':  7,
})

print(f"Mesa version: {mesa.__version__}")
print(f"NumPy version: {np.__version__}")
print("V11 imports complete")


## Model Definition

The V11 model extends earlier versions with:
1. **CPU idle tracking** at every simulation step
2. **Per-run latency means** for standard deviation computation
3. **Peak idle tracking** for worst-case analysis


In [ ]:
class QueueAgent(mesa.Agent):
    """
    Agent representing a transaction in the queue.
    
    Behavioral coefficient (type_coeff) determines
    movement probability and cell occupancy tolerance.
    """
    def __init__(self, model, agent_type):
        super().__init__(model)
        self.agent_type  = agent_type
        self.type_coeff  = TYPE_COEFFICIENTS[agent_type]
        self.urgency     = np.random.uniform(0.3, 1.0)
        self.social_inhibition = np.random.uniform(0.2, 0.8)
        self.exited      = False
        self.entry_time  = None
        self.latency     = None

    @property
    def behavior_score(self):
        """
        Composite behavioral score combining type coefficient,
        urgency, social inhibition, and local density.
        Higher score = higher movement probability.
        """
        base = (self.type_coeff * self.urgency /
                self.social_inhibition)
        if self.pos:
            neighbors = self.model.grid.get_neighbors(
                self.pos, moore=True,
                include_center=False, radius=2
            )
            density = 1.0 + len(neighbors) * 0.05
        else:
            density = 1.0
        return base * density

    def step(self):
        if self.exited or self.pos is None:
            return
        if self.entry_time is None:
            self.entry_time = self.model.steps

        x, y = self.pos
        move_prob = min(self.behavior_score, 1.0)

        if np.random.random() < move_prob:
            new_y = y - 1
            if new_y < 0:
                # Agent exits queue
                self.model.grid.remove_agent(self)
                self.exited  = True
                self.latency = (self.model.steps -
                                self.entry_time)
                self.model.exited_count += 1
                self.model.record_agent(self)
                self.model.moves_this_step += 1
                return
            new_pos = (x, new_y)
            contents = (
                self.model.grid
                .get_cell_list_contents([new_pos])
            )
            # Occupancy limit by aggression level
            max_occ = (
                3 if self.agent_type == AgentType.AGGRESSIVE
                else 2 if self.agent_type == AgentType.NORMAL
                else 1
            )
            if len(contents) < max_occ:
                self.model.grid.move_agent(
                    self, new_pos
                )
                self.model.moves_this_step += 1


class BunchQueueModel(mesa.Model):
    """
    Bunch Queue simulation model with CPU idle tracking.
    
    Parameters
    ----------
    n_agents : int
        Total number of agents
    width, height : int
        Grid dimensions
    pct_aggressive : float
        Fraction of aggressive agents (0.0-1.0)
    pct_normal : float
        Fraction of normal agents (0.0-1.0)
        Remainder are passive agents
    """
    def __init__(self, n_agents=100,
                 width=20, height=30,
                 pct_aggressive=0.15,
                 pct_normal=0.25):
        super().__init__()
        self.width        = width
        self.height       = height
        self.steps        = 0
        self.exited_count = 0
        self.total_agents = n_agents
        self.moves_this_step = 0

        # CPU idle tracking
        self.idle_history  = []
        self.util_history  = []

        # Latency tracking by agent type
        self.latencies = {
            AgentType.PASSIVE:    [],
            AgentType.NORMAL:     [],
            AgentType.AGGRESSIVE: [],
        }

        self.grid = mesa.space.MultiGrid(
            width, height, torus=False
        )

        # Initialize agents
        n_aggr = int(n_agents * pct_aggressive)
        n_norm = int(n_agents * pct_normal)
        n_pass = n_agents - n_aggr - n_norm

        agent_types = (
            [AgentType.AGGRESSIVE] * n_aggr +
            [AgentType.NORMAL]     * n_norm +
            [AgentType.PASSIVE]    * n_pass
        )
        np.random.shuffle(agent_types)

        for agent_type in agent_types:
            agent = QueueAgent(self, agent_type)
            x = np.random.randint(0, width)
            y = np.random.randint(
                height // 2, height
            )
            self.grid.place_agent(agent, (x, y))

    def record_agent(self, agent):
        if agent.latency is not None:
            self.latencies[
                agent.agent_type
            ].append(agent.latency)

    def step(self):
        self.steps += 1
        self.moves_this_step = 0

        self.agents.shuffle_do("step")

        # Compute CPU idle this step
        active = (self.total_agents -
                  self.exited_count)
        if active > 0:
            util = min(
                self.moves_this_step / active, 1.0
            )
            idle = 1.0 - util
        else:
            util = 1.0
            idle = 0.0

        self.idle_history.append(idle)
        self.util_history.append(util)

    def run(self, max_steps=500):
        for _ in range(max_steps):
            self.step()
            if self.exited_count >= self.total_agents:
                break
        return self.exited_count

    def get_idle_stats(self):
        """Returns (avg_idle, peak_idle, avg_util)"""
        if not self.idle_history:
            return 0, 0, 0
        return (
            np.mean(self.idle_history),
            np.max(self.idle_history),
            np.mean(self.util_history),
        )


print("Model defined successfully")
print(f"Agent types: {[t.name for t in AgentType]}")
print(f"Type coefficients: {TYPE_COEFFICIENTS}")


## Experiment Configuration

10 aggression levels × 5 scales × 60 runs = **3,000 simulations per trial**

Three independent trials confirmed reproducibility within 1-3%.


In [ ]:
# ── Aggression Scenarios ─────────────────────
ALL_SCENARIOS = [
    {'pct_agg': 0.05, 'pct_norm': 0.20, 'name': 'Ord 5%'},
    {'pct_agg': 0.15, 'pct_norm': 0.25, 'name': 'Bunch 15%'},
    {'pct_agg': 0.30, 'pct_norm': 0.25, 'name': 'High 30%'},
    {'pct_agg': 0.60, 'pct_norm': 0.15, 'name': '60%'},
    {'pct_agg': 0.65, 'pct_norm': 0.15, 'name': '65%'},
    {'pct_agg': 0.75, 'pct_norm': 0.15, 'name': '75%'},
    {'pct_agg': 0.80, 'pct_norm': 0.10, 'name': '80%'},
    {'pct_agg': 0.85, 'pct_norm': 0.10, 'name': '85%'},
    {'pct_agg': 0.90, 'pct_norm': 0.05, 'name': '90%'},
    {'pct_agg': 0.95, 'pct_norm': 0.04, 'name': '95%'},
]

# ── Scale Configurations ──────────────────────
SCALE_CONFIGS = [
    {'name': '100',  'n': 100,  'w': 20,
     'h': 30,  'steps': 500},
    {'name': '500',  'n': 500,  'w': 45,
     'h': 65,  'steps': 1000},
    {'name': '1000', 'n': 1000, 'w': 65,
     'h': 90,  'steps': 1500},
    {'name': '2500', 'n': 2500, 'w': 100,
     'h': 140, 'steps': 2000},
    {'name': '5000', 'n': 5000, 'w': 140,
     'h': 200, 'steps': 3000},
]

N_RUNS = 60  # Runs per scenario

total = len(ALL_SCENARIOS) * len(SCALE_CONFIGS) * N_RUNS
print(f"Scenarios:  {len(ALL_SCENARIOS)} aggression levels")
print(f"Scales:     {len(SCALE_CONFIGS)}")
print(f"Runs each:  {N_RUNS}")
print(f"Total runs: {total:,}")
print()
print("Aggression levels:")
for s in ALL_SCENARIOS:
    print(f"  {s['name']:<12} "
          f"aggr={s['pct_agg']:.0%} "
          f"norm={s['pct_norm']:.0%} "
          f"pass={1-s['pct_agg']-s['pct_norm']:.0%}")


## Main Experiment

Runs the comprehensive sweep. Auto-saves results after each scale completes.

**Expected runtime:** 2-3 hours for all 5 scales.

Output is simultaneously logged to:
- Jupyter cell output
- `/home/jc/v11_clean_output.txt` (text file)
- `/home/jc/v11_clean_results.json` (JSON data)


In [ ]:
# ── Tee Logger — simultaneous screen and file output ──
class Tee:
    def __init__(self, filename):
        self.file   = open(filename, 'w')
        self.stdout = sys.stdout
    def write(self, text):
        self.file.write(text)
        self.stdout.write(text)
        self.file.flush()
    def flush(self):
        self.file.flush()
        self.stdout.flush()

sys.stdout = Tee('/home/jc/v11_clean_output.txt')

print("V11 COMPREHENSIVE AGGRESSION SWEEP")
print("="*60)
print(f"Scales:     {[s['name'] for s in SCALE_CONFIGS]}")
print(f"Aggression: {[s['name'] for s in ALL_SCENARIOS]}")
print(f"Runs each:  {N_RUNS}")
print("="*60)

start_time = time.time()
v11_results = {}

for scale in SCALE_CONFIGS:
    print(f"\nScale: {scale['name']} agents")
    print("-"*60)
    v11_results[scale['name']] = {}

    for scenario in ALL_SCENARIOS:
        passive_lats      = []
        aggr_lats         = []
        avg_idles         = []
        avg_utils         = []
        throughputs       = []
        run_passive_means = []

        for run in range(N_RUNS):
            model = BunchQueueModel(
                n_agents      = scale['n'],
                width         = scale['w'],
                height        = scale['h'],
                pct_aggressive= scenario['pct_agg'],
                pct_normal    = scenario['pct_norm'],
            )
            cleared = model.run(
                max_steps=scale['steps']
            )
            avg_idle, peak_idle, avg_util = (
                model.get_idle_stats()
            )

            throughputs.append(cleared)
            avg_idles.append(avg_idle)
            avg_utils.append(avg_util)

            if model.latencies[AgentType.PASSIVE]:
                rm = np.mean(
                    model.latencies[AgentType.PASSIVE]
                )
                run_passive_means.append(rm)
                passive_lats.extend(
                    model.latencies[AgentType.PASSIVE]
                )
            if model.latencies[AgentType.AGGRESSIVE]:
                aggr_lats.extend(
                    model.latencies[AgentType.AGGRESSIVE]
                )

        avg_passive = (np.mean(passive_lats)
                      if passive_lats else 0)
        std_passive = (np.std(run_passive_means)
                      if run_passive_means else 0)
        avg_aggr    = (np.mean(aggr_lats)
                      if aggr_lats else 0)
        mean_idle   = np.mean(avg_idles)
        mean_util   = np.mean(avg_utils)
        mean_thru   = np.mean(throughputs)

        v11_results[scale['name']][
            scenario['name']
        ] = {
            'passive':    avg_passive,
            'std':        std_passive,
            'aggressive': avg_aggr,
            'idle':       mean_idle,
            'util':       mean_util,
            'throughput': mean_thru,
        }

        elapsed = (time.time() - start_time) / 60
        print(
            f"  {scenario['name']:<12} "
            f"Passive:{avg_passive:7.1f} "
            f"(±{std_passive:.1f})  "
            f"Idle:{mean_idle*100:5.1f}%  "
            f"Util:{mean_util*100:5.1f}%  "
            f"[{elapsed:.0f}min]"
        )

    # Auto-save after each scale
    with open('/home/jc/v11_clean_results.json',
              'w') as f:
        json.dump(v11_results, f, indent=2)
    elapsed = (time.time() - start_time) / 60
    print(f"\n  Scale {scale['name']} complete "
          f"[{elapsed:.0f}min] — saved")

elapsed_total = (time.time() - start_time) / 60
print(f"\n{'='*60}")
print(f"V11 COMPLETE — {elapsed_total:.0f} minutes")
print(f"{'='*60}")
print(f"Results saved to v11_clean_results.json")


## Summary Tables

Print formatted summary tables from the completed results.


In [ ]:
# ── Passive Latency Summary ──────────────────
print("PASSIVE LATENCY SUMMARY (simulation steps)")
print(f"{'Scale':<8}", end="")
for s in ALL_SCENARIOS:
    print(f" {s['name']:>10}", end="")
print(f" {'Best':>8}")
print("-"*100)

for scale in SCALE_CONFIGS:
    print(f"{scale['name']:<8}", end="")
    best_lat  = float('inf')
    best_name = ""
    for s in ALL_SCENARIOS:
        r = v11_results[scale['name']][s['name']]
        print(f" {r['passive']:>10.1f}", end="")
        if r['passive'] < best_lat:
            best_lat  = r['passive']
            best_name = s['name']
    print(f" {best_name:>8}")

print()

# ── CPU Idle Summary ──────────────────────────
print("CPU IDLE RATE SUMMARY (%)")
print(f"{'Scale':<8}", end="")
for s in ALL_SCENARIOS:
    print(f" {s['name']:>10}", end="")
print(f" {'Best':>8}")
print("-"*100)

for scale in SCALE_CONFIGS:
    print(f"{scale['name']:<8}", end="")
    best_idle = float('inf')
    best_name = ""
    for s in ALL_SCENARIOS:
        r = v11_results[scale['name']][s['name']]
        print(f" {r['idle']*100:>9.1f}%", end="")
        if r['idle'] < best_idle:
            best_idle = r['idle']
            best_name = s['name']
    print(f" {best_name:>8}")

print()

# ── Standard Deviation Summary ────────────────
print("STANDARD DEVIATION SUMMARY (steps)")
print(f"{'Scale':<8}", end="")
for s in ALL_SCENARIOS:
    print(f" {s['name']:>10}", end="")
print()
print("-"*100)

for scale in SCALE_CONFIGS:
    print(f"{scale['name']:<8}", end="")
    for s in ALL_SCENARIOS:
        r = v11_results[scale['name']][s['name']]
        print(f" {r['std']:>10.1f}", end="")
    print()

print()

# ── Key improvement vs orderly ────────────────
print("IMPROVEMENT VS ORDERLY BASELINE (5%)")
print(f"{'Scale':<8} {'Orderly':>10} {'Best':>10} "
      f"{'At':>8} {'Improvement':>12}")
print("-"*55)

for scale in SCALE_CONFIGS:
    orderly = v11_results[
        scale['name']]['Ord 5%']['passive']
    best = float('inf')
    best_name = ""
    for s in ALL_SCENARIOS:
        r = v11_results[
            scale['name']][s['name']]['passive']
        if r < best:
            best = r
            best_name = s['name']
    improvement = (orderly - best) / orderly * 100
    print(f"{scale['name']:<8} "
          f"{orderly:>10.1f} "
          f"{best:>10.1f} "
          f"{best_name:>8} "
          f"{improvement:>11.1f}%")


## Publication Figures

Generate the four key figures from V11 results.

- **Fig 2** — Passive latency vs aggression level
- **Fig 3** — CPU idle rate vs aggression level  
- **Fig 4** — Standard deviation (thrashing signal)
- **Fig 7** — Improvement by scale (two-panel)


In [ ]:
# ── Prepare data arrays ──────────────────────
AGG_LEVELS  = [5, 15, 30, 60, 65, 75, 80, 85, 90, 95]
SCALE_NAMES = ['100', '500', '1000', '2500', '5000']

COLORS = {
    '100':  '#000000',
    '500':  '#333333',
    '1000': '#666666',
    '2500': '#999999',
    '5000': '#BBBBBB',
}
MARKERS = {
    '100': 'o', '500': 's',
    '1000': '^', '2500': 'D', '5000': 'v',
}
LINES = {
    '100':  '-',    '500':  '--',
    '1000': '-.',   '2500': ':',
    '5000': (0,(5,1)),
}

passive_data = {}
idle_data    = {}
std_data     = {}

for scale in SCALE_NAMES:
    passive_data[scale] = []
    idle_data[scale]    = []
    std_data[scale]     = []
    for s in ALL_SCENARIOS:
        r = v11_results[scale][s['name']]
        passive_data[scale].append(r['passive'])
        idle_data[scale].append(r['idle'] * 100)
        std_data[scale].append(r['std'])

print("Data arrays prepared")
print(f"Scales: {SCALE_NAMES}")
print(f"Aggression levels: {AGG_LEVELS}")


In [ ]:
# ── Fig 2 — Passive Latency ──────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))
fig.subplots_adjust(bottom=0.18)

for scale in SCALE_NAMES:
    ax.plot(
        AGG_LEVELS, passive_data[scale],
        color=COLORS[scale],
        marker=MARKERS[scale],
        linestyle=LINES[scale],
        label=f'{scale} agents',
        linewidth=2, markersize=6,
    )

ax.axvspan(75, 90, alpha=0.08, color='black',
           label='Optimal zone')
ax.text(81, ax.get_ylim()[0] +
        (ax.get_ylim()[1]-ax.get_ylim()[0])*0.15,
        'Optimal\nzone\n75-90%',
        fontsize=8, ha='center',
        style='italic', color='#444444')

ax.set_xlabel('Aggression Level (%)')
ax.set_ylabel('Passive Agent Latency (simulation steps)',
              fontsize=9)
ax.set_title(
    'Passive Agent Latency vs Aggression Level'
)
ax.set_xticks(AGG_LEVELS)
ax.set_xticklabels(
    [str(x) for x in AGG_LEVELS],
    fontsize=8, rotation=45
)
ax.legend(fontsize=7, loc='upper left',
          ncol=2, borderpad=0.4,
          labelspacing=0.3,
          handlelength=1.5,
          columnspacing=0.8)

plt.savefig('/home/jc/Fig2_PassiveLatency.png',
            dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("Fig 2 saved: Fig2_PassiveLatency.png")


In [ ]:
# ── Fig 3 — CPU Idle Rate ────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))
fig.subplots_adjust(bottom=0.18)

for scale in SCALE_NAMES:
    ax.plot(
        AGG_LEVELS, idle_data[scale],
        color=COLORS[scale],
        marker=MARKERS[scale],
        linestyle=LINES[scale],
        label=f'{scale} agents',
        linewidth=2, markersize=6,
    )

ax.axvspan(75, 90, alpha=0.08, color='black')
ax.text(81, 52,
        'Optimal\nzone',
        fontsize=8, ha='center',
        style='italic', color='#444444')

ax.set_xlabel('Aggression Level (%)')
ax.set_ylabel('CPU Idle Rate (%)', fontsize=9)
ax.set_title('CPU Idle Rate vs Aggression Level')
ax.set_xticks(AGG_LEVELS)
ax.set_xticklabels(
    [str(x) for x in AGG_LEVELS],
    fontsize=8, rotation=45
)
ax.set_ylim(40, 82)
ax.legend(fontsize=7, loc='upper right',
          ncol=2, borderpad=0.4,
          labelspacing=0.3,
          handlelength=1.5,
          columnspacing=0.8)

plt.savefig('/home/jc/Fig3_CPUIdle.png',
            dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("Fig 3 saved: Fig3_CPUIdle.png")


In [ ]:
# ── Fig 4 — Standard Deviation ───────────────
fig, ax = plt.subplots(figsize=(7, 4.5))
fig.subplots_adjust(bottom=0.18)

for scale in SCALE_NAMES:
    ax.plot(
        AGG_LEVELS, std_data[scale],
        color=COLORS[scale],
        marker=MARKERS[scale],
        linestyle=LINES[scale],
        label=f'{scale} agents',
        linewidth=2, markersize=6,
    )

ax.axvline(x=75, color='black',
           linestyle=':', linewidth=1.5,
           alpha=0.6, label='Thrashing onset')
ax.text(76, ax.get_ylim()[1]*0.85,
        'Std dev triples\nabove 75%',
        fontsize=8, color='#444444',
        style='italic')

ax.set_xlabel('Aggression Level (%)')
ax.set_ylabel('Std Dev of Passive Latency (steps)',
              fontsize=9)
ax.set_title(
    'Standard Deviation vs Aggression Level '
    '— Thrashing Signal'
)
ax.set_xticks(AGG_LEVELS)
ax.set_xticklabels(
    [str(x) for x in AGG_LEVELS],
    fontsize=8, rotation=45
)
ax.legend(fontsize=7, loc='upper left',
          ncol=2, borderpad=0.4,
          labelspacing=0.3,
          handlelength=1.5,
          columnspacing=0.8)

plt.savefig('/home/jc/Fig4_StdDev.png',
            dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("Fig 4 saved: Fig4_StdDev.png")


In [ ]:
# ── Fig 7 — Two-Panel Improvement by Scale ───
orderly_passive = {
    s['name']: v11_results[
        sc['name']][s['name']]['passive']
    for sc in SCALE_CONFIGS
    for s in ALL_SCENARIOS
    if s['name'] == 'Ord 5%'
}
orderly_passive = {
    sc['name']: v11_results[
        sc['name']]['Ord 5%']['passive']
    for sc in SCALE_CONFIGS
}
orderly_idle = {
    sc['name']: v11_results[
        sc['name']]['Ord 5%']['idle'] * 100
    for sc in SCALE_CONFIGS
}

# Best at optimal aggression
best_improvement = [15, 20, 23, 26, 26]
best_idle_reduction = [26, 16, 13, 12, 11]
scale_labels = ['100', '500', '1000',
                '2500', '5000']

fig, axes = plt.subplots(
    1, 2, figsize=(10, 4.5)
)
fig.subplots_adjust(wspace=0.35, bottom=0.18)

# Panel (a) — Latency improvement
ax1 = axes[0]
bars1 = ax1.bar(scale_labels, best_improvement,
                color='#555555',
                edgecolor='black', linewidth=0.8)
ax1.bar_label(bars1,
              labels=[f'{v}%' for v in best_improvement],
              padding=3, fontsize=9)
ax1.set_xlabel('Scale (agents)')
ax1.set_ylabel('Improvement vs Orderly (%)',
               fontsize=9)
ax1.set_title('(a) Latency Improvement at
Optimal Aggression (85-90%)')
ax1.set_ylim(0, 32)

# Panel (b) — CPU idle reduction
ax2 = axes[1]
bars2 = ax2.bar(scale_labels, best_idle_reduction,
                color='#888888',
                edgecolor='black', linewidth=0.8)
ax2.bar_label(bars2,
              labels=[f'{v}pp' for v in best_idle_reduction],
              padding=3, fontsize=9)
ax2.set_xlabel('Scale (agents)')
ax2.set_ylabel('CPU Idle Reduction (percentage points)',
               fontsize=9)
ax2.set_title('(b) CPU Idle Reduction at
Optimal Aggression (85-90%)')
ax2.set_ylim(0, 32)

plt.savefig('/home/jc/Fig7_ImprovementScale.png',
            dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("Fig 7 saved: Fig7_ImprovementScale.png")


## Pre-Validated Results

The following cell loads the three-run validated results confirmed across
three independent trials. Use these as the authoritative dataset for the paper.

These results are saved in `v11_clean_results.json` and
reflect the average of three complete 60-run trials.


In [ ]:
# ── Three-Run Validated Averages ─────────────
# These are the confirmed results from three
# independent 60-run trials
# Reproducibility: within 1-3% across all runs

VALIDATED_RESULTS = {
    'passive_latency': {
        '100':  [151.3, 151.2, 146.2, 138.6, 134.0,
                 131.7, 129.6, 128.0, 130.7, 128.1],
        '500':  [356.7, 353.3, 341.3, 309.6, 304.0,
                 288.7, 292.3, 292.5, 289.8, 284.2],
        '1000': [518.4, 511.4, 497.4, 450.1, 434.8,
                 407.3, 411.2, 397.2, 399.7, 407.8],
        '2500': [854.2, 843.6, 825.4, 742.2, 714.7,
                 651.6, 649.5, 643.5, 633.1, 639.5],
        '5000': [1273.6, 1260.0, 1238.7, 1114.0, 1072.2,
                 958.4, 955.1, 949.5, 940.7, 947.0],
    },
    'cpu_idle_pct': {
        '100':  [73.3, 72.9, 69.8, 62.7, 60.4,
                 51.6, 51.5, 47.6, 50.8, 48.7],
        '500':  [76.5, 75.5, 73.9, 70.0, 68.0,
                 61.7, 62.5, 61.7, 62.3, 61.3],
        '1000': [77.4, 76.5, 75.1, 71.3, 69.6,
                 64.5, 65.3, 64.2, 64.7, 64.5],
        '2500': [78.4, 77.5, 76.4, 72.5, 71.7,
                 67.6, 67.9, 67.4, 66.5, 67.2],
        '5000': [79.1, 78.3, 77.1, 73.6, 72.5,
                 69.1, 69.3, 69.0, 69.1, 69.1],
    },
    'std_dev': {
        '100':  [13.9, 12.7, 16.6, 19.1, 23.7,
                 34.6, 35.6, 38.9, 34.1, 32.7],
        '500':  [13.7, 15.2, 15.7, 21.4, 24.2,
                 33.6, 33.2, 35.6, 33.6, 37.9],
        '1000': [13.5, 13.9, 15.9, 20.5, 22.6,
                 38.6, 36.6, 38.0, 38.3, 32.3],
        '2500': [14.6, 15.5, 16.5, 21.8, 26.0,
                 39.0, 38.0, 35.2, 37.3, 36.1],
        '5000': [14.1, 15.8, 20.3, 29.4, 27.9,
                 30.8, 39.4, 33.0, 38.5, 41.0],
    },
    'aggression_levels': [5,15,30,60,65,75,80,85,90,95],
    'scales': ['100','500','1000','2500','5000'],
    'n_runs': 60,
    'n_trials': 3,
    'notes': (
        'Three-run validated averages. '
        'Results reproducible within 1-3% '
        'across independent trials.'
    )
}

print("Three-run validated results loaded")
print(f"Scales: {VALIDATED_RESULTS['scales']}")
print(f"Aggression levels: "
      f"{VALIDATED_RESULTS['aggression_levels']}")
print(f"Runs per scenario: {VALIDATED_RESULTS['n_runs']}")
print(f"Independent trials: {VALIDATED_RESULTS['n_trials']}")
print()
print("Key finding — 5000 agents at optimal 90%:")
idx_90 = VALIDATED_RESULTS[
    'aggression_levels'].index(90)
orderly_5k = VALIDATED_RESULTS[
    'passive_latency']['5000'][0]
optimal_5k = VALIDATED_RESULTS[
    'passive_latency']['5000'][idx_90]
improvement = (orderly_5k - optimal_5k) / orderly_5k * 100
print(f"  Orderly baseline: {orderly_5k:.1f} steps")
print(f"  Optimal (90%):    {optimal_5k:.1f} steps")
print(f"  Improvement:      {improvement:.1f}%")
